# 91. CHARGEMENT DES DONNÉES

On charge le fichier nettoyé par Zouhair (87 208 réservations)

Contrôle : on doit obtenir (87208, 31) = 87 208 lignes, 31 colonnes

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics


In [2]:
df = pd.read_csv("../data/hotel_bookings_clean.csv")
print(df.shape)

(87208, 31)


**Interprétation** : le fichier nettoyé contient 87 208 réservations et 31 colonnes, sans valeur manquante. C'est le même nombre que le nettoyage SQL : Python et SQL travaillent sur les mêmes données.

# 2. CHOIX DES VARIABLES (ANTI-LEAKAGE)

Moment de la prédiction : le jour où la réservation est faite. On ne garde que les informations connues à ce moment-là.

**Variables gardées (18)**
- 10 numériques : délai de réservation, nuits, nombre de personnes, client fidèle, historique d'annulations, prix moyen par nuit.
- 8 catégorielles : hôtel, mois d'arrivée, formule repas, segment de marché, canal de vente, type de chambre réservé, type d'acompte, type de client.

**Variables exclues**
- reservation_status et reservation_status_date : elles contiennent la réponse (fuite de données).
- assigned_room_type, booking_changes, days_in_waiting_list : connues seulement après la réservation.
- total_of_special_requests, required_car_parking_spaces : le client peut les ajouter après la réservation.
- country : souvent complétée à l'arrivée du client.
- agent et company : ce sont des identifiants, pas des quantités.

X = les variables que le modèle regarde pour décider. y = la cible à prédire (1 = annulée, 0 = non annulée).

Contrôle : on doit obtenir (87208, 18) et 27,51 % d'annulations.

In [3]:
num_cols = [
    "lead_time",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adults",
    "children",
    "babies",
    "is_repeated_guest",
    "previous_cancellations",
    "previous_bookings_not_canceled",
    "adr",
]

cat_cols = [
    "hotel",
    "arrival_date_month",
    "meal",
    "market_segment",
    "distribution_channel",
    "reserved_room_type",
    "deposit_type",
    "customer_type",
]

X = df[num_cols + cat_cols]
y = df["is_canceled"]

print(X.shape)
print(round(y.mean() * 100, 2))

(87208, 18)
27.51


**Interprétation** : le modèle dispose de 18 variables connues au moment de la réservation. 27,51 % des réservations sont annulées : les classes sont déséquilibrées (environ 1 annulation pour 3 réservations honorées), ce qui rend l'accuracy seule trompeuse.

# 3. ENCODAGE DES VARIABLES CATÉGORIELLES

Un modèle ne comprend que des nombres. On transforme chaque variable catégorielle en colonnes 0/1 avec pd.get_dummies.

Exemple : hotel devient hotel_City Hotel (0 ou 1) et hotel_Resort Hotel (0 ou 1).

On garde toutes les catégories (pas de drop_first), pour que chaque catégorie reste lisible dans l’interprétation des modèles. 

In [4]:
X_enc = pd.get_dummies(X, columns=cat_cols, dtype=int)

print(X_enc.shape)
X_enc.head()

(87208, 55)


,lead_time,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,adr,...,reserved_room_type_G,reserved_room_type_H,reserved_room_type_L,deposit_type_No Deposit,deposit_type_Non Refund,deposit_type_Refundable,customer_type_Contract,customer_type_Group,customer_type_Transient,customer_type_Transient-Party
0,342,0,0,2,0.0,0,0,0,0,0.0,...,0,0,0,1,0,0,0,0,1,0
1,737,0,0,2,0.0,0,0,0,0,0.0,...,0,0,0,1,0,0,0,0,1,0
2,7,0,1,1,0.0,0,0,0,0,75.0,...,0,0,0,1,0,0,0,0,1,0
3,13,0,1,1,0.0,0,0,0,0,75.0,...,0,0,0,1,0,0,0,0,1,0
4,14,0,2,2,0.0,0,0,0,0,98.0,...,0,0,0,1,0,0,0,0,1,0


**Interprétation** : on passe de 18 à 55 colonnes. Les 10 variables numériques restent 10 colonnes. Les 8 variables catégorielles deviennent 45 colonnes 0/1, une par catégorie (12 pour les mois, 7 pour les segments de marché...).

# 4. SÉPARATION ENTRAÎNEMENT / TEST ET STANDARDISATION

- train_test_split : 80 % pour l'entraînement, 20 % pour le test. stratify=y garde 27,51 % d'annulations dans chaque partie.
- StandardScaler : appliqué **après** le split, sur les 10 variables numériques seulement (les colonnes 0/1 restent telles quelles).
- fit_transform sur l'entraînement, transform sur le test, pour éviter la fuite de données.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X_enc, y, test_size=0.2, random_state=42, stratify=y
)

scaler_std = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[num_cols] = scaler_std.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler_std.transform(X_test[num_cols])

print(X_train_scaled.shape, X_test_scaled.shape)
print(round(y_train.mean() * 100, 2))
X_train_scaled.describe()

(69766, 55) (17442, 55)
27.51


,lead_time,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,adr,...,reserved_room_type_G,reserved_room_type_H,reserved_room_type_L,deposit_type_No Deposit,deposit_type_Non Refund,deposit_type_Refundable,customer_type_Contract,customer_type_Group,customer_type_Transient,customer_type_Transient-Party
count,6.976600e+04,6.976600e+04,6.976600e+04,6.976600e+04,6.976600e+04,6.976600e+04,6.976600e+04,6.976600e+04,6.976600e+04,6.976600e+04,...,69766.000000,69766.000000,69766.000000,69766.000000,69766.000000,69766.000000,69766.000000,69766.000000,69766.000000,69766.000000
mean,-2.016562e-17,-5.133067e-17,-4.542357e-17,-1.185494e-16,-8.962498e-18,-1.527698e-17,3.340567e-17,-3.157243e-18,9.471730e-18,1.986008e-16,...,0.023693,0.006823,0.000043,0.987114,0.011668,0.001218,0.035848,0.005948,0.823739,0.134464
std,1.000007e+00,1.000007e+00,1.000007e+00,1.000007e+00,1.000007e+00,1.000007e+00,1.000007e+00,1.000007e+00,1.000007e+00,1.000007e+00,...,0.152094,0.082319,0.006557,0.112783,0.107385,0.034884,0.185913,0.076897,0.381044,0.341152
min,-9.286592e-01,-9.795469e-01,-1.289927e+00,-3.710836e+00,-3.048931e-01,-1.026078e-01,-1.996002e-01,-7.946903e-02,-1.065740e-01,-2.053011e+00,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,-8.008496e-01,-9.795469e-01,-7.975372e-01,2.432857e-01,-3.048931e-01,-1.026078e-01,-1.996002e-01,-7.946903e-02,-1.065740e-01,-6.614681e-01,...,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
50%,-3.593254e-01,-3.386047e-03,-3.051479e-01,2.432857e-01,-3.048931e-01,-1.026078e-01,-1.996002e-01,-7.946903e-02,-1.065740e-01,-1.539641e-01,...,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
75%,5.237229e-01,9.727748e-01,6.796308e-01,2.432857e-01,-3.048931e-01,-1.026078e-01,-1.996002e-01,-7.946903e-02,-1.065740e-01,5.336220e-01,...,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
max,7.634586e+00,1.659135e+01,1.939043e+01,1.605977e+01,6.317513e+00,1.893605e+01,5.010016e+00,6.631858e+01,4.146335e+01,7.769647e+00,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


**Interprétation** : 69 766 réservations servent à l'entraînement (80 %) et 17 442 au test (20 %). La part d'annulations reste de 27,51 % dans l'entraînement grâce à stratify. Après standardisation, les 10 variables numériques ont une moyenne proche de 0 et un écart-type de 1 : lead_time (jusqu'à 737 jours) ne pèse plus davantage qu'adults (jusqu'à 10) à cause de son échelle.

# 5. RÉFÉRENCE : PRÉDIRE "PAS D'ANNULATION" POUR TOUT LE MONDE

Avant d'entraîner un vrai modèle, on mesure le score d'une règle très simple : prédire "non annulée" pour toutes les réservations du test.

Tout modèle doit faire mieux que ce score, sinon il ne sert à rien.

In [6]:
print((y_test == 0).mean())

0.724916867331728


**Interprétation** : en prédisant "non annulée" pour toutes les réservations, on obtient 72,5 % de bonnes réponses. C'est le score minimum à dépasser.

72,5 % des réservations du test ne sont pas annulées. Une règle qui répond toujours "non" a donc raison 72,5 % du temps, sans rien détecter. Une accuracy de 75 % paraîtra bonne, mais ne représentera qu'un faible progrès : il faudra regarder d'autres mesures (rappel, AUC).

In [24]:
from sklearn.dummy import DummyClassifier

modele_dummy = DummyClassifier(strategy="most_frequent")
modele_dummy.fit(X_train_scaled, y_train)

y_pred_dummy = modele_dummy.predict(X_test_scaled)
proba_dummy = modele_dummy.predict_proba(X_test_scaled)[:, 1]

print("Accuracy :", round(metrics.accuracy_score(y_test, y_pred_dummy), 4))
print("AUC :", round(metrics.roc_auc_score(y_test, proba_dummy), 4))

Accuracy : 0.7249
AUC : 0.5


Le DummyClassifier (stratégie "most_frequent") confirme ce résultat : 72,5 % d'accuracy, mais une AUC de 0,5, c'est-à-dire aucune capacité à classer les réservations par risque. C'est le niveau minimum que les modèles devront dépasser, en particulier sur l'AUC.

# 6. ENTRAÎNEMENT DES MODÈLES

On entraîne 3 modèles sur les données d'entraînement (X_train_scaled, y_train) :

- **Régression logistique** : modèle linéaire qui calcule une probabilité d'appartenance à une classe avec une fonction sigmoïde. 
- **Arbre de décision** : divise les données par une suite de règles (exemple : lead_time > 90 ? puis deposit_type = Non Refund ?). Limité à 5 niveaux (max_depth=5) pour rester lisible et éviter le surapprentissage.
- **Random Forest** : combine 100 arbres entraînés sur des sous-échantillons et fait voter leurs prédictions. Plus stable qu'un arbre seul, mais moins lisible.

random_state=42 rend les résultats reproductibles.

In [7]:
modele_lr = LogisticRegression(max_iter=1000, random_state=42)
modele_lr.fit(X_train_scaled, y_train)

modele_tree = DecisionTreeClassifier(max_depth=5, random_state=42)
modele_tree.fit(X_train_scaled, y_train)

modele_rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
modele_rf.fit(X_train_scaled, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


# 7. PRÉDICTIONS SUR LE JEU DE TEST

Chaque modèle prédit, pour les 17 442 réservations du test qu'il n'a jamais vues, si elles seront annulées (1) ou non (0).

On compare ensuite les prédictions (y_pred) aux vraies valeurs (y_test) :
- **accuracy** : part de prédictions correctes ;
- **nombre d'annulations prédites**, comparé aux 4 798 vraies annulations du test.

In [8]:
y_pred_lr = modele_lr.predict(X_test_scaled)
y_pred_tree = modele_tree.predict(X_test_scaled)
y_pred_rf = modele_rf.predict(X_test_scaled)

print("Vraies annulations dans le test :", y_test.sum())

print("Régression logistique :", (y_pred_lr == y_test).mean(), "| annulations prédites :", y_pred_lr.sum())
print("Arbre de décision :", (y_pred_tree == y_test).mean(), "| annulations prédites :", y_pred_tree.sum())
print("Random Forest :", (y_pred_rf == y_test).mean(), "| annulations prédites :", y_pred_rf.sum())

Vraies annulations dans le test : 4798
Régression logistique : 0.7544432977869511 | annulations prédites : 1421
Arbre de décision : 0.7463593624584337 | annulations prédites : 386
Random Forest : 0.7512899896800825 | annulations prédites : 572


**Interprétation** : les 3 modèles obtiennent entre 74,6 % et 75,4 % de bonnes réponses, soit seulement 2 à 3 points de plus que la référence (72,5 %). L'accuracy ne montre presque aucun progrès.

Le vrai problème apparaît dans le nombre d'annulations prédites : le test contient 4 798 annulations, mais la régression logistique n'en prédit que 1 421, le Random Forest 572 et l'arbre 386. Les modèles répondent "non annulée" dans la grande majorité des cas, comme la référence : ils passent à côté de la plupart des annulations.

# 8. INTERPRÉTATION DES MODÈLES

Quelles variables influencent le plus la prédiction d'annulation ?

- **Régression logistique** : chaque variable a un coefficient. Positif = augmente le risque d'annulation. Négatif = le diminue. Plus il est grand (en valeur absolue), plus l'effet est fort.
- **Random Forest** : feature_importances_ indique la contribution de chaque variable aux décisions des arbres (total = 1). Elle donne l'intensité, pas le sens.

In [9]:
feature_names = X_train_scaled.columns
coefficients = modele_lr.coef_[0]

coef_lr = pd.DataFrame({"variable": feature_names, "coefficient": coefficients})
coef_lr = coef_lr.sort_values("coefficient")

print(coef_lr.head(8))
print(coef_lr.tail(8))

                          variable  coefficient
48         deposit_type_No Deposit    -1.627116
50         deposit_type_Refundable    -0.861435
33    market_segment_Offline TA/TO    -0.806935
8   previous_bookings_not_canceled    -0.628797
52             customer_type_Group    -0.348544
44            reserved_room_type_F    -0.272725
31           market_segment_Direct    -0.233753
36     distribution_channel_Direct    -0.216724
                        variable  coefficient
34      market_segment_Online TA     0.250577
38    distribution_channel_TA/TO     0.253067
28       market_segment_Aviation     0.304626
29  market_segment_Complementary     0.372525
0                      lead_time     0.405454
53       customer_type_Transient     0.606748
7         previous_cancellations     0.677383
49       deposit_type_Non Refund     2.422085


**Interprétation (régression logistique)** :

Un coefficient positif augmente le risque d'annulation, un coefficient négatif le diminue.

Ce qui **augmente** le risque :
- deposit_type_Non Refund (+2,42) : le facteur le plus fort ;
- previous_cancellations (+0,68) : un client qui a déjà annulé annule plus souvent (requête R7) ;
- customer_type_Transient (+0,61) : les réservations individuelles annulent plus ;
- lead_time (+0,41) : plus on réserve tôt, plus on annule (requête R3) ;
- market_segment_Online TA (+0,25) : les agences en ligne annulent plus (requête R2).

Ce qui **diminue** le risque :
- deposit_type_No Deposit (−1,63) et Refundable (−0,86) ;
- market_segment_Offline TA/TO (−0,81) ;
- previous_bookings_not_canceled (−0,63) : les clients fidèles sont fiables (requête R7).

Prudence :
- Non Refund : 94,7 % de ces réservations sont annulées, ce qui va contre l'intuition. C'est une particularité connue de ce jeu de données (probablement des blocs de chambres réservés par des agences puis libérés). Elle ne concerne que 1 037 réservations sur 87 208.
- Complementary (+0,37) et Aviation (+0,30) : coefficients peu fiables, car ces segments comptent très peu de réservations.

In [10]:
importances = modele_rf.feature_importances_

imp_rf = pd.DataFrame({"variable": feature_names, "importance": importances})
imp_rf = imp_rf.sort_values("importance", ascending=False)

print(imp_rf.head(10))

                         variable  importance
0                       lead_time    0.211331
34       market_segment_Online TA    0.118877
7          previous_cancellations    0.087949
48        deposit_type_No Deposit    0.074666
49        deposit_type_Non Refund    0.065445
33   market_segment_Offline TA/TO    0.055964
9                             adr    0.053209
53        customer_type_Transient    0.051638
54  customer_type_Transient-Party    0.036826
2            stays_in_week_nights    0.030026


**Interprétation (Random Forest)** :

L'importance indique l'intensité de l'effet, pas son sens. Les 3 variables les plus importantes sont :
1. lead_time (0,211) : de loin la première, elle pèse 2 fois plus que la suivante ;
2. market_segment_Online TA (0,119) ;
3. previous_cancellations (0,088).

Suivent deposit_type (No Deposit 0,075, Non Refund 0,065), Offline TA/TO (0,056) et adr (0,053).

# 9. ÉVALUATION DES MODÈLES

On se concentre sur la classe 1 (annulation), celle qui intéresse la direction.

- **Matrice de confusion** : [[vrais négatifs, faux positifs], [faux négatifs, vrais positifs]].
- **Précision** : quand le modèle annonce une annulation, a-t-il raison ?
- **Rappel** : sur toutes les vraies annulations, combien en détecte-t-il ?
- **F1-score** : équilibre entre précision et rappel.
- **AUC** : capacité à classer les réservations de la plus risquée à la moins risquée (0,5 = hasard, 1 = parfait). Elle utilise les probabilités (predict_proba), pas les prédictions 0/1.

In [11]:
modeles = {
    "Régression logistique": (modele_lr, y_pred_lr),
    "Arbre de décision": (modele_tree, y_pred_tree),
    "Random Forest": (modele_rf, y_pred_rf),
}

for nom, (modele, y_pred) in modeles.items():
    proba = modele.predict_proba(X_test_scaled)[:, 1]
    print(nom)
    print(metrics.confusion_matrix(y_test, y_pred))
    print("Précision :", round(metrics.precision_score(y_test, y_pred), 3))
    print("Rappel :", round(metrics.recall_score(y_test, y_pred), 3))
    print("F1 :", round(metrics.f1_score(y_test, y_pred), 3))
    print("AUC :", round(metrics.roc_auc_score(y_test, proba), 4))
    print()

Régression logistique
[[12191   453]
 [ 3830   968]]
Précision : 0.681
Rappel : 0.202
F1 : 0.311
AUC : 0.7483

Arbre de décision
[[12638     6]
 [ 4418   380]]
Précision : 0.984
Rappel : 0.079
F1 : 0.147
AUC : 0.7487

Random Forest
[[12588    56]
 [ 4282   516]]
Précision : 0.902
Rappel : 0.108
F1 : 0.192
AUC : 0.7761



**Lecture de la matrice de confusion**

|  | Prédit : non annulée (0) | Prédit : annulée (1) |
|---|---|---|
| **Réel : non annulée (0)** | Vrais négatifs (bonne réponse) | Faux positifs (fausse alerte) |
| **Réel : annulée (1)** | Faux négatifs (annulation ratée) | Vrais positifs (annulation détectée) |

**Régression logistique**

|  | Prédit : non annulée | Prédit : annulée | Total |
|---|---|---|---|
| **Réel : non annulée** | 12 191 | 453 | 12 644 |
| **Réel : annulée** | 3 830 | 968 | 4 798 |

**Arbre de décision**

|  | Prédit : non annulée | Prédit : annulée | Total |
|---|---|---|---|
| **Réel : non annulée** | 12 638 | 6 | 12 644 |
| **Réel : annulée** | 4 418 | 380 | 4 798 |

**Random Forest**

|  | Prédit : non annulée | Prédit : annulée | Total |
|---|---|---|---|
| **Réel : non annulée** | 12 588 | 56 | 12 644 |
| **Réel : annulée** | 4 282 | 516 | 4 798 |

**Interprétation** :

Les 3 modèles sont très prudents : quand ils annoncent une annulation, ils ont souvent raison (précision de 68 % à 98 %), mais ils en détectent très peu (rappel de 8 % à 20 %).

Exemple, le Random Forest : sur 4 798 annulations réelles du test, il en détecte 516 (vrais positifs) et en rate 4 282 (faux négatifs). Il ne se trompe que 56 fois en annonçant une annulation (faux positifs).

- Faux négatif : une annulation non anticipée, donc une chambre qui risque de rester vide.
- Faux positif : un client qui serait venu est contacté inutilement.

Ce faible rappel vient du seuil par défaut de 0,5 : le modèle ne prédit "annulée" que lorsqu'il est presque sûr.

Mais la direction ne cherche pas à prédire toutes les annulations. Elle veut **classer** les réservations pour choisir les 100 plus risquées sur 1 000. La bonne mesure est donc l'**AUC**, qui évalue ce classement.

Les 3 modèles obtiennent une AUC de 0,748 à 0,776, nettement supérieure à celle du DummyClassifier (0,5) : contrairement au modèle naïf, ils savent classer les réservations par risque.

On retient le **Random Forest**, qui a la meilleure AUC (0,776, contre 0,748 pour la régression logistique et 0,749 pour l'arbre de décision). On n'utilisera pas le seuil de 0,5 mais le classement par probabilité (étape 10).

# 9.1 RÉGLAGE DU RANDOM FOREST (max_depth)

On teste plusieurs valeurs de max_depth (profondeur maximale des arbres) et on compare les scores.

On affiche l'AUC sur l'entraînement et sur le test : un grand écart entre les deux signale du surapprentissage.

In [12]:
for profondeur in [5, 10, 15, 20, 25, None]:
    modele = RandomForestClassifier(n_estimators=100, max_depth=profondeur, random_state=42)
    modele.fit(X_train_scaled, y_train)
    auc_train = metrics.roc_auc_score(y_train, modele.predict_proba(X_train_scaled)[:, 1])
    auc_test = metrics.roc_auc_score(y_test, modele.predict_proba(X_test_scaled)[:, 1])
    print("max_depth =", profondeur, "| AUC train :", round(auc_train, 3), "| AUC test :", round(auc_test, 4))

max_depth = 5 | AUC train : 0.751 | AUC test : 0.7561
max_depth = 10 | AUC train : 0.786 | AUC test : 0.7761
max_depth = 15 | AUC train : 0.839 | AUC test : 0.7903
max_depth = 20 | AUC train : 0.913 | AUC test : 0.7963
max_depth = 25 | AUC train : 0.975 | AUC test : 0.7936
max_depth = None | AUC train : 0.999 | AUC test : 0.7715


**Interprétation** : plus l'arbre est profond, plus le score d'entraînement monte (jusqu'à 0,999 sans limite). Le score de test progresse jusqu'à max_depth = 20 (0,796), puis baisse : au-delà, le modèle apprend par cœur les réservations d'entraînement et généralise moins bien (surapprentissage).

On retient max_depth = 20 : l'AUC passe de 0,776 à 0,796.

On choisit la profondeur qui donne le meilleur score sur des données non vues, pas sur l'entraînement. La plus grande profondeur donne 0,999 sur l'entraînement mais seulement 0,772 sur le test.

**Limite** : le réglage a été choisi en regardant le test. Le score de 0,796 est donc légèrement optimiste : une partie de validation séparée donnerait une estimation plus prudente.

# 9.2 RÉGLAGE DE L'ARBRE DE DÉCISION (max_depth)

On teste plusieurs profondeurs maximales pour l'arbre de décision et on compare l'AUC sur l'entraînement et sur le test.

In [14]:
for profondeur in [3, 5, 8, 10, 12, 15, None]:
    modele = DecisionTreeClassifier(max_depth=profondeur, random_state=42)
    modele.fit(X_train_scaled, y_train)
    auc_train = metrics.roc_auc_score(y_train, modele.predict_proba(X_train_scaled)[:, 1])
    auc_test = metrics.roc_auc_score(y_test, modele.predict_proba(X_test_scaled)[:, 1])
    print("max_depth =", profondeur, "| AUC train :", round(auc_train, 3), "| AUC test :", round(auc_test, 4))

max_depth = 3 | AUC train : 0.716 | AUC test : 0.7195
max_depth = 5 | AUC train : 0.746 | AUC test : 0.7487
max_depth = 8 | AUC train : 0.766 | AUC test : 0.7627
max_depth = 10 | AUC train : 0.781 | AUC test : 0.7662
max_depth = 12 | AUC train : 0.8 | AUC test : 0.764
max_depth = 15 | AUC train : 0.837 | AUC test : 0.7563
max_depth = None | AUC train : 1.0 | AUC test : 0.6483


**Interprétation** : le score de test progresse jusqu'à max_depth = 10 (0,766), puis baisse. Sans limite de profondeur, l'arbre obtient un score parfait sur l'entraînement (1,000) mais s'effondre sur le test (0,648) : il a appris par cœur les réservations d'entraînement. C'est un cas net de surapprentissage.

On retient max_depth = 10 : l'AUC passe de 0,749 à 0,766. L'arbre reste moins performant que le Random Forest (0,796), qui combine 100 arbres et limite ainsi le surapprentissage.

# 9.3 RÉGLAGE DE LA RÉGRESSION LOGISTIQUE (C)

Le paramètre C règle la force de la régularisation, qui empêche le modèle de trop s'ajuster aux données d'entraînement :
- C petit : régularisation forte, modèle plus simple ;
- C grand : régularisation faible, modèle plus libre.

La valeur par défaut est C = 1. On teste plusieurs valeurs et on compare l'AUC sur l'entraînement et sur le test.

In [16]:
for valeur_c in [0.01, 0.1, 1, 10, 100]:
    modele = LogisticRegression(C=valeur_c, max_iter=2000, random_state=42)
    modele.fit(X_train_scaled, y_train)
    auc_train = metrics.roc_auc_score(y_train, modele.predict_proba(X_train_scaled)[:, 1])
    auc_test = metrics.roc_auc_score(y_test, modele.predict_proba(X_test_scaled)[:, 1])
    print("C =", valeur_c, "| AUC train :", round(auc_train, 3), "| AUC test :", round(auc_test, 4))

C = 0.01 | AUC train : 0.74 | AUC test : 0.7468
C = 0.1 | AUC train : 0.741 | AUC test : 0.7481
C = 1 | AUC train : 0.742 | AUC test : 0.7483
C = 10 | AUC train : 0.742 | AUC test : 0.7483
C = 100 | AUC train : 0.742 | AUC test : 0.7483


**Interprétation** : le paramètre C ne change presque rien : l'AUC de test reste entre 0,747 et 0,748, quelle que soit la valeur. Les scores d'entraînement et de test sont très proches : le modèle ne surapprend pas.

La régression logistique est donc déjà à son maximum avec le réglage par défaut (C = 1). Sa limite ne vient pas de son réglage mais de sa nature : elle ne capte que des relations linéaires (chaque variable augmente ou diminue le risque de façon constante). Les modèles à base d'arbres captent des effets plus complexes, comme des seuils ou des combinaisons de variables.

On garde C = 1. La régression logistique reste utile pour interpréter le sens de chaque variable (étape 8), mais elle n'est pas le modèle le plus performant.

# 9.4 BILAN DES RÉGLAGES ET MODÈLE FINAL


| Modèle | AUC test avant réglage | Meilleur réglage | AUC test après réglage |
|---|---|---|---|
| DummyClassifier (référence) | 0,5 | – | 0,5 |
| Régression logistique | 0,748 | C = 1 (défaut) | 0,748 |
| Arbre de décision | 0,749 | max_depth = 10 | 0,766 |
| **Random Forest** | 0,776 | max_depth = 20 | **0,796** |

On retient le Random Forest avec max_depth = 20. On l'entraîne une dernière fois et on mesure ses performances complètes sur le test.

In [19]:
modele_rf_final = RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42)
modele_rf_final.fit(X_train_scaled, y_train)

y_pred_final = modele_rf_final.predict(X_test_scaled)
proba_final = modele_rf_final.predict_proba(X_test_scaled)[:, 1]

print(metrics.confusion_matrix(y_test, y_pred_final))
print("Précision :", round(metrics.precision_score(y_test, y_pred_final), 3))
print("Rappel :", round(metrics.recall_score(y_test, y_pred_final), 3))
print("F1 :", round(metrics.f1_score(y_test, y_pred_final), 3))
print("AUC :", round(metrics.roc_auc_score(y_test, proba_final), 4))

[[12154   490]
 [ 3560  1238]]
Précision : 0.716
Rappel : 0.258
F1 : 0.379
AUC : 0.7963


**Interprétation** : après réglage, le Random Forest reste le meilleur des 3 modèles, avec une avance plus nette (AUC de 0,796, contre 0,766 pour l'arbre et 0,748 pour la régression logistique).

Par rapport au Random Forest de départ (max_depth = 10) :
- l'AUC passe de 0,776 à 0,796 : il classe mieux les réservations par risque ;
- il détecte 1 238 annulations au lieu de 516 (rappel de 26 % au lieu de 11 %) ;
- en contrepartie, la précision baisse de 90 % à 72 % : 490 fausses alertes au lieu de 56.

Le modèle plus profond capte davantage de situations d'annulation. Il est moins "prudent" au seuil de 0,5, mais surtout meilleur pour classer les réservations, ce qui compte pour la décision de l'étape 10.

# 10. DÉCISION : QUELLES RÉSERVATIONS CIBLER ?

Contrainte de la direction : sur 1 000 réservations à venir, l'équipe peut agir sur 100 seulement (relance, demande d'acompte, confirmation).

Règle : classer les réservations par probabilité d'annulation (Random Forest final) et cibler les 100 premières.

Simulation : on tire 1 000 réservations au hasard dans le test, on applique la règle, puis on compte combien des 100 réservations ciblées ont réellement été annulées.

In [20]:
test = df.loc[X_test.index, ["hotel", "deposit_type"]].copy()
test["proba"] = proba_final
test["annule"] = y_test

lot = test.sample(1000, random_state=42)
top100 = lot.nlargest(100, "proba")

print("Taux d'annulation moyen des 1 000 :", round(lot["annule"].mean() * 100, 1), "%")
print("Annulations réelles parmi les 100 ciblées :", top100["annule"].sum())
print(top100["deposit_type"].value_counts())

Taux d'annulation moyen des 1 000 : 30.3 %
Annulations réelles parmi les 100 ciblées : 75
deposit_type
No Deposit    82
Non Refund    18
Name: count, dtype: int64


**Interprétation** : sur 1 000 réservations, 30,3 % sont annulées. Un ciblage au hasard toucherait donc environ 30 annulations sur 100 réservations contactées. Avec le classement du Random Forest, **75 des 100 réservations ciblées ont réellement été annulées** : la règle est 2,5 fois plus efficace que le hasard.

Parmi ces 100 réservations, 18 ont un acompte non remboursable : l'hôtel garde déjà l'argent en cas d'annulation, les contacter a donc peu d'intérêt.

Utilisé pour prioriser, le modèle multiplie par 2,5 l'efficacité des actions de l'équipe.

**Limite** : ce résultat repose sur un seul tirage de 1 000 réservations.

# 10.1 VÉRIFICATIONS DE LA RÈGLE

Deux vérifications complètent la décision :

1. **Écarter les acomptes non remboursables** : l'hôtel garde déjà l'argent si ces réservations sont annulées. On cible donc les 100 réservations les plus risquées parmi les autres.
2. **Tester sur tout le jeu de test** : au lieu d'un seul tirage de 1 000 réservations, on prend les 10 % les plus risquées des 17 442 réservations du test (même proportion que 100 sur 1 000).

In [22]:
top100_sans_nr = lot[lot["deposit_type"] != "Non Refund"].nlargest(100, "proba")
print("Sans Non Refund : annulations parmi les 100 ciblées :", top100_sans_nr["annule"].sum())

nb_cibles = len(test) // 10
top10_pct = test.nlargest(nb_cibles, "proba")
print("Réservations ciblées (10 % du test) :", nb_cibles)
print("Taux d'annulation des 10 % ciblées :", round(top10_pct["annule"].mean() * 100, 1), "%")
print("Taux d'annulation moyen du test :", round(test["annule"].mean() * 100, 1), "%")

Sans Non Refund : annulations parmi les 100 ciblées : 68
Réservations ciblées (10 % du test) : 1744
Taux d'annulation des 10 % ciblées : 71.6 %
Taux d'annulation moyen du test : 27.5 %


**Interprétation** :

1. **Sans les acomptes non remboursables**, 68 des 100 réservations ciblées sont réellement annulées. C'est un peu moins que 75, mais ces 68 annulations représentent toutes une vraie perte potentielle pour l'hôtel. C'est la règle la plus utile pour la direction.

2. **Sur tout le jeu de test**, parmi les 10 % de réservations les plus risquées (1 744), 71,6 % sont réellement annulées, contre 27,5 % en moyenne. Le résultat est confirmé sur 17 442 réservations : la règle est environ 2,6 fois plus efficace qu'un ciblage au hasard.

**Réponse à la question** : oui. Le résultat ne dépend pas d'un tirage chanceux : il se vérifie sur l'ensemble du test.

**Recommandation** : chaque semaine, classer les réservations à venir par probabilité d'annulation, écarter celles avec acompte non remboursable, puis contacter les 100 premières sur 1 000 (relance, reconfirmation ou demande d'acompte).

# 11. VARIANTE : CIBLER SELON LE RISQUE × LA VALEUR

Toutes les annulations ne coûtent pas la même chose. Annuler une nuit à 60 € n'a pas le même impact qu'annuler une semaine à 150 € la nuit.

On calcule pour chaque réservation :
- **montant** = prix moyen par nuit (adr) × nombre de nuits ;
- **risque × valeur** = probabilité d'annulation × montant, soit le chiffre d'affaires "en danger".

On compare deux règles pour choisir les 100 réservations à cibler (hors acomptes non remboursables) :
1. classement par probabilité d'annulation (étape 10) ;
2. classement par risque × valeur.


In [23]:
nuits = df.loc[lot.index, "stays_in_weekend_nights"] + df.loc[lot.index, "stays_in_week_nights"]
lot["montant"] = df.loc[lot.index, "adr"] * nuits
lot["risque_valeur"] = lot["proba"] * lot["montant"]

lot_sans_nr = lot[lot["deposit_type"] != "Non Refund"]
ca_annule_total = (lot_sans_nr["montant"] * lot_sans_nr["annule"]).sum()
print("CA annulé total des réservations (hors Non Refund) :", round(ca_annule_total), "€")

for regle in ["proba", "risque_valeur"]:
    cibles = lot_sans_nr.nlargest(100, regle)
    ca_annule_cible = (cibles["montant"] * cibles["annule"]).sum()
    print(regle, "| annulations ciblées :", cibles["annule"].sum(),
          "| CA annulé ciblé :", round(ca_annule_cible), "€",
          "| part du CA annulé :", round(ca_annule_cible / ca_annule_total * 100, 1), "%")

CA annulé total des réservations (hors Non Refund) : 140417 €
proba | annulations ciblées : 68 | CA annulé ciblé : 56917 € | part du CA annulé : 40.5 %
risque_valeur | annulations ciblées : 60 | CA annulé ciblé : 69095 € | part du CA annulé : 49.2 %


# CONCLUSION

**Question métier** : peut-on anticiper, au moment de la réservation, qu'elle sera annulée, et en faire une règle d'action ?

**Modèle retenu** : Random Forest (max_depth = 20), AUC de 0,796 sur le test. Il fait mieux que la régression logistique (0,748) et l'arbre de décision (0,766).

**Facteurs principaux** : le délai de réservation (lead_time), les annulations passées du client, le segment des agences en ligne et le type d'acompte. Ils confirment l'analyse SQL.

**Règle de décision** : chaque semaine, classer les réservations à venir par probabilité d'annulation, écarter celles avec acompte non remboursable, puis contacter les 100 premières sur 1 000. Sur le jeu de test, les 10 % de réservations les plus risquées sont annulées à 71,6 %, contre 27,5 % en moyenne : la règle est environ 2,6 fois plus efficace qu'un ciblage au hasard.

**Variante recommandée pour le contrôle de gestion** : classer selon le risque × la valeur (probabilité d'annulation × montant du séjour). Avec le même nombre d'actions, cette règle cible 49,2 % du chiffre d'affaires annulé, contre 40,5 % avec la seule probabilité, soit environ 12 000 € de plus protégés sur 1 000 réservations.

**Limites** :
- données de 2015 à 2017, sur deux hôtels portugais : les comportements ont pu changer ;
- découpage aléatoire entre entraînement et test, alors que le taux d'annulation augmente dans le temps : un découpage temporel serait plus réaliste ;
- réglages choisis en regardant le test : le score de 0,796 est légèrement optimiste ;
- le modèle ne dit pas si l'action (relance, acompte) changera réellement le comportement du client ;
- particularité des acomptes non remboursables, très souvent annulés dans ces données.

**Interprétation** : les deux règles ne visent pas le même objectif.

- **Classement par probabilité** : il touche le plus d'annulations (68 sur 100), mais ces réservations représentent 56 917 € de chiffre d'affaires annulé, soit 40,5 % du total perdu.
- **Classement par risque × valeur** : il touche un peu moins d'annulations (60 sur 100), mais elles représentent 69 095 €, soit 49,2 % du chiffre d'affaires perdu. Avec le même nombre d'actions, l'équipe protège environ 12 000 € de plus.

Ce résultat prolonge la requête SQL R4 : les réservations annulées valent plus que la moyenne. Il est donc rentable de prioriser les séjours longs ou chers.

**Limites** : simulation sur un seul tirage de 1 000 réservations ; le montant est une estimation (prix moyen × nuits) qui ne tient pas compte des chambres revendues après annulation.